# Implementing Knowledge Distillation

This notebook walks through a complete implementation of knowledge distillation with working code examples. We'll build everything from scratch and train a real student model.

## Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm


torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

## Step 1: Temperature Scaling Function

Temperature controls how "smooth" the probability distribution becomes. Let's implement and test it:

In [ ]:
def apply_temperature(logits, temperature):
    """
    temperature scaled softmax, the soft targets distillation trains on
    
    args:
        logits: raw model outputs before softmax
        temperature: higher gives a softer distribution, 1 is the ordinary softmax
    
    returns:
        softened probability distribution
    """
    return F.softmax(logits / temperature, dim=-1)

# same example logits as notebook 01
logits = torch.tensor([10.0, 2.0, 1.0, 0.5])

print("temp scaling examples:")
print("="*60)
for T in [1, 3, 5, 10]:
    probs = apply_temperature(logits, temperature=T)
    print(f"\nT={T:2d}: {probs.numpy()}")
    print(f"Entropy: {-(probs * torch.log(probs + 1e-10)).sum():.4f}")

print("\nHigher temperature = Higher entropy = More uncertainty revealed")

## Step 2: The Distillation Loss Function

The core of knowledge distillation - combining soft targets from the teacher with hard labels:

In [ ]:
def distillation_loss(student_logits, teacher_logits, labels, temperature=3.0, alpha=0.7):
    """
    knowledge distillation loss after hinton, vinyals and dean (2015)
    
    args:
        student_logits: raw outputs from the student
        teacher_logits: raw outputs from the frozen teacher
        labels: ground truth class indices
        temperature: softening parameter, above 1 exposes how the teacher ranks wrong classes (default: 3.0)
        alpha: weight on the distillation term, 1 - alpha goes to the label term (default: 0.7)
    
    returns:
        (total, distill, student) so the two terms can be logged separately
    """
    # kl_div expects the student as log probabilities and the teacher as probabilities
    soft_teacher = F.softmax(teacher_logits / temperature, dim=-1)
    soft_student = F.log_softmax(student_logits / temperature, dim=-1)
    
    # distillation term: kl(teacher_t || student_t), batchmean gives the per example kl
    # times t^2 because soft target gradients scale as 1 / t^2, this keeps both terms balanced as t changes
    distill_loss = F.kl_div(
        soft_student, 
        soft_teacher, 
        reduction='batchmean') * (temperature ** 2)
    
    # label term: ordinary cross entropy at t=1 against the ground truth
    student_loss = F.cross_entropy(student_logits, labels)
    
    total_loss = alpha * distill_loss + (1 - alpha) * student_loss
    
    return total_loss, distill_loss, student_loss

# smoke test on random logits
print("Testing Distillation Loss Function:")


batch_size = 4
num_classes = 10
dummy_student = torch.randn(batch_size, num_classes)
dummy_teacher = torch.randn(batch_size, num_classes)
dummy_labels = torch.randint(0, num_classes, (batch_size,))

total, distill, student = distillation_loss(dummy_student, dummy_teacher, dummy_labels, temperature=3.0, alpha=0.7)

print(f"Total Loss: {total:.4f}")
print(f"Distillation Loss: {distill:.4f} (weight: 0.7)")
print(f"Student Loss: {student:.4f} (weight: 0.3)")
print(f"\nVerify: {0.7*distill + 0.3*student:.4f} = {total:.4f}")

## Step 3: Define Teacher and Student Models

Let's create realistic models for image classification (e.g., MNIST-style):

In [ ]:
class TeacherModel(nn.Module):
    """large mlp teacher, about 13x the student's parameters"""
    def __init__(self, input_size=784, num_classes=10):
        super(TeacherModel, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        return self.network(x)

class StudentModel(nn.Module):
    """small mlp student, the model meant for deployment"""
    def __init__(self, input_size=784, num_classes=10):
        super(StudentModel, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x):
        return self.network(x)

teacher = TeacherModel().to(device)
student = StudentModel().to(device)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

teacher_params = count_parameters(teacher)
student_params = count_parameters(student)

print("model architecture comparison:")
print(f"Teacher parameters: {teacher_params:,}")
print(f"Student parameters: {student_params:,}")
print(f"Compression ratio: {teacher_params/student_params:.1f}x")
print(f"Size reduction: {(1-student_params/teacher_params)*100:.1f}%")

print("\nTeacher Architecture:")
print(teacher)
print("\nStudent Architecture:")
print(student)

## Step 4: Create Synthetic Dataset

For this example, we'll create synthetic data to demonstrate the training process:

In [ ]:
def create_synthetic_data(n_samples=1000, input_size=784, num_classes=10):
    """
    random features with independently random labels: there is no signal to learn,
    so test accuracy stays at chance (about 10%) and only the training mechanics are shown
    """
    X = torch.randn(n_samples, input_size)
    y = torch.randint(0, num_classes, (n_samples,))
    
    return X, y

train_X, train_y = create_synthetic_data(n_samples=2000)
test_X, test_y = create_synthetic_data(n_samples=500)

train_dataset = TensorDataset(train_X, train_y)
test_dataset = TensorDataset(test_X, test_y)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Batch size: {train_loader.batch_size}")
print(f"Number of batches: {len(train_loader)}")

## Step 5: Pre-train the Teacher Model

First, we need a well-trained teacher. Let's train it on our data:

In [ ]:
# plain supervised training on hard labels, reused below for the no distillation control
def train_teacher(model, train_loader, num_epochs=15, lr=0.001):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    model.train()
    
    losses = []
    
    print("Training Teacher Model...")
    for epoch in range(num_epochs):
        epoch_loss = 0
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = F.cross_entropy(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        avg_loss = epoch_loss / len(train_loader)
        losses.append(avg_loss)
        
        if (epoch + 1) % 3 == 0:
            print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")
    
    return losses

teacher_losses = train_teacher(teacher, train_loader, num_epochs=15)

plt.figure(figsize=(10, 5))
plt.plot(teacher_losses, marker='o', linewidth=2, markersize=6)
plt.title('Teacher Model Training Loss', fontsize=14, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nFinal teacher loss: {teacher_losses[-1]:.4f}")

## Step 6: Train Student with Distillation

Now train the student model using knowledge distillation:

In [ ]:
def train_student_distillation(student_model, teacher_model, train_loader, num_epochs=15, temperature=4.0, alpha=0.7, lr=0.001):
    """
    train the student on the frozen teacher's softened outputs plus the true labels
    """
    optimizer = optim.Adam(student_model.parameters(), lr=lr)
    teacher_model.eval()  # dropout off, so the soft targets are deterministic
    
    losses = []
    distill_losses = []
    student_losses = []
    
    print(f"Training Student with Distillation (T={temperature}, α={alpha})...")
    
    for epoch in range(num_epochs):
        student_model.train()
        epoch_loss = 0
        epoch_distill = 0
        epoch_student = 0
        
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            
            # no_grad: the teacher is frozen, only the student's optimizer steps
            with torch.no_grad():
                teacher_logits = teacher_model(batch_X)
            
            student_logits = student_model(batch_X)
            
            total_loss, distill_loss, stud_loss = distillation_loss(student_logits, teacher_logits, batch_y,temperature=temperature, alpha=alpha)
            
            total_loss.backward()
            optimizer.step()
            
            epoch_loss += total_loss.item()
            epoch_distill += distill_loss.item()
            epoch_student += stud_loss.item()
        
        avg_loss = epoch_loss / len(train_loader)
        avg_distill = epoch_distill / len(train_loader)
        avg_student = epoch_student / len(train_loader)
        
        losses.append(avg_loss)
        distill_losses.append(avg_distill)
        student_losses.append(avg_student)
        
        if (epoch + 1) % 3 == 0:
            print(f"Epoch {epoch+1}/{num_epochs} | "
                  f"Total: {avg_loss:.4f} | "
                  f"Distill: {avg_distill:.4f} | "
                  f"Student: {avg_student:.4f}")
    
    return losses, distill_losses, student_losses

student_losses, distill_losses, hard_losses = train_student_distillation(student, teacher, train_loader,num_epochs=15, temperature=4.0, alpha=0.7)

## Step 7: Visualize Training Progress

Let's visualize how the different loss components evolved:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(teacher_losses, label='Teacher Training', marker='o', linewidth=2)
axes[0].plot(student_losses, label='Student Distillation', marker='s', linewidth=2)
axes[0].set_title('Total Training Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(distill_losses, label='Distillation Loss (from teacher)', 
             marker='^', linewidth=2, color='orange')
axes[1].plot(hard_losses, label='Student Loss (from labels)', 
             marker='v', linewidth=2, color='green')
axes[1].plot(student_losses, label='Total Loss', 
             marker='s', linewidth=2, color='blue', linestyle='--')
axes[1].set_title('Student Loss Components', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Notice how the student learns from both the teacher (distillation loss)")
print("and the true labels (student loss)!")

## Step 8: Evaluate and Compare Models

Let's evaluate both models on the test set:

In [ ]:
# top 1 test accuracy at t=1, argmax is unaffected by temperature anyway
def evaluate_model(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            predictions = torch.argmax(outputs, dim=1)
            correct += (predictions == batch_y).sum().item()
            total += batch_y.size(0)
    
    accuracy = 100 * correct / total
    return accuracy

teacher_acc = evaluate_model(teacher, test_loader)
student_acc = evaluate_model(student, test_loader)

print("Model Evaluation Results:")
print(f"Teacher Accuracy:{teacher_acc:.2f}%")
print(f"Student Accuracy: {student_acc:.2f}%")
print(f"Accuracy Gap: {teacher_acc - student_acc:.2f}%")
print(f"Performance Retained: {(student_acc/teacher_acc)*100:.1f}%")
print(f"\nModel Size Comparison:")
print(f"Teacher params:{teacher_params:,}")
print(f"Student params:{student_params:,}")
print(f"Compression: {teacher_params/student_params:.1f}x smaller")

## Step 9: Compare with Baseline Student

Let's train a student from scratch (without distillation) to see the benefit:

In [ ]:
# control: same student architecture, data and epochs, hard labels only
baseline_student = StudentModel().to(device)

print("Training Baseline student (without distillation)...")
baseline_losses = train_teacher(baseline_student, train_loader, num_epochs=15)

baseline_acc = evaluate_model(baseline_student, test_loader)
print("Final Comparison:")
print(f"Teacher (large model):{teacher_acc:.2f}%")
print(f"Student with Distillation: {student_acc:.2f}%")
print(f"Student without Distillation:s{baseline_acc:.2f}%")
print(f"\nDistillation Improvement: +{student_acc - baseline_acc:.2f}%")
print(f"Gap to Teacher (with distillation): {teacher_acc - student_acc:.2f}%")
print(f"Gap to Teacher (baseline):{teacher_acc - baseline_acc:.2f}%")

## Step 10: Hyperparameter Exploration

Let's experiment with different temperatures and alpha values:

In [ ]:
# t and alpha change together in each config, so their effects cannot be separated
configs = [
    {'temperature': 2.0, 'alpha': 0.5},
    {'temperature': 4.0, 'alpha': 0.7},
    {'temperature': 6.0, 'alpha': 0.9},]

results = []

print("hyperparameter exploration:")


for config in configs:
    test_student = StudentModel().to(device)
    
    print(f"\nTesting T={config['temperature']}, α={config['alpha']}")
    _, _, _ = train_student_distillation(
        test_student, teacher, train_loader,
        num_epochs=10,
        temperature=config['temperature'],
        alpha=config['alpha'],
        lr=0.001)
    
    acc = evaluate_model(test_student, test_loader)
    results.append({
        'Temperature': config['temperature'],
        'Alpha': config['alpha'],
        'Accuracy': acc})
    print(f"Result: {acc:.2f}%")

import pandas as pd
df_results = pd.DataFrame(results)
print("Results Summary:")
print(df_results.to_string(index=False))
print("\nBest configuration:", df_results.loc[df_results['Accuracy'].idxmax()].to_dict())

## Key Takeaways

1. **Distillation works**: Students trained with distillation outperform those trained from scratch
2. **Temperature matters**: Higher temperature reveals more information but can be too soft
3. **Alpha balances**: Need to balance between teacher knowledge and true labels
4. **Compression achieved**: Much smaller models with minimal accuracy loss

## Hyperparameter Guidelines

- **Temperature (T)**: Start with 3-5, increase if student struggles, decrease if too different from teacher
- **Alpha (α)**: Start with 0.7, increase to rely more on teacher, decrease for more label supervision
- **Learning rate**: Often same or slightly higher than teacher's original LR
